In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [2]:
import csv
import psutil
import numpy as np
import numba as nb
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

from _load_dataset import load_dataset_sparse_labels

In [3]:
logical_cores = os.cpu_count()
physical_cores = psutil.cpu_count(logical=False)

print("Logical cores:", logical_cores)
print("Physical cores:", physical_cores or "unknown")

Logical cores: 36
Physical cores: 18


In [4]:
SEED_RANGE = (0, 1000)
TRAIN_RATIO1 = 0.8
TEST_RATIO2 = 0.2
N_COMPONENTS = 3

DIRECTION = "maximize" # "maximize" or "minimize"
OUT_DIR = "runs/pca_seed"

NUM_WORKERS = physical_cores // 2

In [5]:
os.makedirs(OUT_DIR, exist_ok=True)

In [6]:
_, s008_lidar, _, _, s009_lidar, _ = load_dataset_sparse_labels()
s008_flat = s008_lidar.reshape(s008_lidar.shape[0], -1)
s009_flat = s009_lidar.reshape(s009_lidar.shape[0], -1)

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [7]:
@nb.njit
def compute_metrics(diff):
    """Compute L1, L2, Linf and MSE for a length-3 diff vector."""
    l1 = abs(diff[0]) + abs(diff[1]) + abs(diff[2])
    sum_sq = diff[0] * diff[0] + diff[1] * diff[1] + diff[2] * diff[2]
    l2 = np.sqrt(sum_sq)
    linf = max(abs(diff[0]), abs(diff[1]), abs(diff[2]))
    mse = sum_sq / 3.0
    return l1, l2, linf, mse


_ = compute_metrics(np.zeros(3, dtype=np.float64))


def pca_split_metrics(X1, X2, split1, split2, seed, n_components, pca_method="pca", batch_size=500):
    """
    Split X1 and X2, fit PCA on each, compute explained-variance-ratio differences.

    Args:
        X1: array of shape (n1, d)
        X2: array of shape (n2, d)
        split1: train_size fraction for X1
        split2: test_size fraction for X2
        seed: random seed
        n_components: number of PCA components
        pca_method: PCA method to use ("pca" or "incremental")

    Returns:
        tuple (l1, l2, linf, mse)
    """
    train1, _ = train_test_split(X1, train_size=split1, random_state=seed)
    _, test2 = train_test_split(X2, test_size=split2, random_state=seed)

    if pca_method == "pca":
        p1 = PCA(n_components=n_components, random_state=seed).fit(train1)
        p2 = PCA(n_components=n_components, random_state=seed).fit(test2)
    elif pca_method == "incremental":
        ipca1 = IncrementalPCA(n_components=n_components)
        for start in range(0, train1.shape[0], batch_size):
            end = start + batch_size
            ipca1.partial_fit(train1[start:end])

        ipca2 = IncrementalPCA(n_components=n_components)
        for start in range(0, test2.shape[0], batch_size):
            end = start + batch_size
            ipca2.partial_fit(test2[start:end])
    else:
        raise ValueError("Invalid PCA method")

    diff = p1.explained_variance_ratio_ - p2.explained_variance_ratio_
    return compute_metrics(diff)


def process_seed(seed):
    l1, l2, linf, mse = pca_split_metrics(
        s008_flat,
        s009_flat,
        TRAIN_RATIO1,
        TEST_RATIO2,
        seed,
        N_COMPONENTS,
        pca_method="pca",
        batch_size=4096
    )
    return seed, l1, l2, linf, mse

In [8]:
csv_path = os.path.join(OUT_DIR, "results.csv")
results = []

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["seed", "l1", "l2", "linf", "mse"])

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for seed, l1, l2, linf, mse in executor.map(process_seed, range(SEED_RANGE[0], SEED_RANGE[1] + 1)):
            # write immediately
            writer.writerow([seed, f"{l1:.6f}", f"{l2:.6f}", f"{linf:.6f}", f"{mse:.6f}"])
            f.flush()

            # in-memory list (for later best‐seed logic)
            results.append((seed, l1, l2, linf, mse))

            # simple status print
            print(f"Processed seed {seed}: L1={l1:.3f}, L2={l2:.3f}, Linf={linf:.3f}, MSE={mse:.6f}")

print(f"Results written to {csv_path}")

results.sort(key=lambda x: x[0])

Processed seed 0: L1=0.126, L2=0.115, Linf=0.115, MSE=0.004416
Processed seed 1: L1=0.127, L2=0.114, Linf=0.114, MSE=0.004362
Processed seed 2: L1=0.132, L2=0.119, Linf=0.118, MSE=0.004719
Processed seed 3: L1=0.133, L2=0.119, Linf=0.119, MSE=0.004747
Processed seed 4: L1=0.144, L2=0.125, Linf=0.124, MSE=0.005212
Processed seed 5: L1=0.140, L2=0.121, Linf=0.121, MSE=0.004907
Processed seed 6: L1=0.137, L2=0.122, Linf=0.122, MSE=0.005001
Processed seed 7: L1=0.136, L2=0.122, Linf=0.121, MSE=0.004946
Processed seed 8: L1=0.142, L2=0.124, Linf=0.124, MSE=0.005154
Processed seed 9: L1=0.144, L2=0.126, Linf=0.125, MSE=0.005258
Processed seed 10: L1=0.139, L2=0.121, Linf=0.120, MSE=0.004887
Processed seed 11: L1=0.141, L2=0.125, Linf=0.125, MSE=0.005240
Processed seed 12: L1=0.132, L2=0.117, Linf=0.116, MSE=0.004525
Processed seed 13: L1=0.137, L2=0.119, Linf=0.119, MSE=0.004754
Processed seed 14: L1=0.152, L2=0.130, Linf=0.129, MSE=0.005664
Processed seed 15: L1=0.144, L2=0.127, Linf=0.127,

In [9]:
print("L1 -> L1 norm, this sums the absolute per-component differences.")
print("L2 -> L2 norm, this is the Euclidean distance between the two vectors.")
print("Linf -> Linf norm, this is the maximum absolute per-component difference.")
print("MSE -> Mean Squared Error, this is the average of the squared differences.\n")

if DIRECTION == "minimize":
    best_l1 = min(results, key=lambda x: x[1])
    best_l2 = min(results, key=lambda x: x[2])
    best_linf = min(results, key=lambda x: x[3])
    best_mse = min(results, key=lambda x: x[4])
else:
    best_l1 = max(results, key=lambda x: x[1])
    best_l2 = max(results, key=lambda x: x[2])
    best_linf = max(results, key=lambda x: x[3])
    best_mse = max(results, key=lambda x: x[4])

best_seed = best_mse[0]
print(f"\nPlotting best seed {best_seed} (MSE={best_mse[4]:.6f})...")

train1, _ = train_test_split(s008_flat, train_size=TRAIN_RATIO1, random_state=best_seed)
_, test2 = train_test_split(s009_flat, test_size=TEST_RATIO2, random_state=best_seed)
p1 = PCA(n_components=N_COMPONENTS, random_state=best_seed).fit(train1)
p2 = PCA(n_components=N_COMPONENTS, random_state=best_seed).fit(test2)
proj1 = p1.transform(train1)
proj2 = p2.transform(test2)

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(proj1[:, 0], proj1[:, 1], proj1[:, 2], label="s008", alpha=0.5)
ax.scatter(proj2[:, 0], proj2[:, 1], proj2[:, 2], label="s009", alpha=0.5)
ax.set_title(
    f"Seed {best_seed}  L1={best_l1[1]:.3f}  L2={best_l2[2]:.3f}  "
    f"Linf={best_linf[3]:.3f}  MSE={best_mse[4]:.6f}"
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.legend()

fname = (
    f"{OUT_DIR}/best_seed_{best_seed}_"
    f"L1_{best_l1[1]:.3f}_"
    f"L2_{best_l2[2]:.3f}_"
    f"Linf_{best_linf[3]:.3f}_"
    f"MSE_{best_mse[4]:.6f}.png"
)
fig.savefig(fname)
plt.close(fig)

print(f"Saved 3D scatter to {fname}\n")
print("Summary:")
print(f"  Best seed for {DIRECTION} L1: {best_l1[0]} (L1 = {best_l1[1]:.3f})")
print(f"  Best seed for {DIRECTION} L2: {best_l2[0]} (L2 = {best_l2[2]:.3f})")
print(f"  Best seed for {DIRECTION} Linf: {best_linf[0]} (Linf = {best_linf[3]:.3f})")
print(f"  Best seed for {DIRECTION} MSE: {best_mse[0]} (MSE = {best_mse[4]:.6f})")

L1 -> L1 norm, this sums the absolute per-component differences.
L2 -> L2 norm, this is the Euclidean distance between the two vectors.
Linf -> Linf norm, this is the maximum absolute per-component difference.
MSE -> Mean Squared Error, this is the average of the squared differences.


Plotting best seed 99 (MSE=0.006279)...
Saved 3D scatter to runs/pca_seed/best_seed_99_L1_0.161_L2_0.137_Linf_0.136_MSE_0.006279.png

Summary:
  Best seed for maximize L1: 99 (L1 = 0.161)
  Best seed for maximize L2: 99 (L2 = 0.137)
  Best seed for maximize Linf: 99 (Linf = 0.136)
  Best seed for maximize MSE: 99 (MSE = 0.006279)
